In [23]:
import os
import re
import time
import string
from pathlib import Path
from typing import List, Tuple, Dict, Any


import numpy as np
import pandas as pd
from PIL import Image, ImageOps


import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST

DATA_ROOT = Path("Cursive")
INVENTORY_CSV = Path("file_inventory.csv")
ALLOWED_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.heic'}
GENERIC_NAME_PREFIXES = ("IMG_", "2024")
STUDENT_ID_REGEX = re.compile(r'^[Ss]\d+$')


BATCH_SIZE = 64
MEAN_STD = ((0.1307,), (0.3081,))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Frame the problem
Using the customer description, Define the problem your trying to solve in your own words (remember this is not technial but must be specific so the customer understands the project

Customer needs a model that can analyze hand written cursive, and predict what character is written.

# 2. Get the Data 
Define how you recieved the data (provided, gathered..)

Data has been given to us by the boss/customer through a google drive link.

# 3. Explore the Data
Gain insights into the data you have from step 2, making sure to identify any bias

Uploaded the data to jupyter as a zip file, then unzipped using unzip, then added folder to .gitignore file to prevent LFS error in github.

In [25]:
if not INVENTORY_CSV.exists():
    records = []
    for fp in DATA_ROOT.glob("**/*"):
        if not fp.is_file():
            continue
        ext = fp.suffix.lower()
        if ext not in ALLOWED_EXTENSIONS:
            continue

        student_id = "UNKNOWN"
        for parent in fp.parents:
            if parent == DATA_ROOT:
                break
            if STUDENT_ID_REGEX.match(parent.name):
                student_id = parent.name
                break

        label_candidate = fp.stem.strip()
        is_generic = any(label_candidate.lower().startswith(p.lower()) for p in GENERIC_NAME_PREFIXES)
        is_single_letter = (not is_generic) and (len(label_candidate) == 1)

        final_png = str(fp.with_suffix('.png'))
        status = 'READY' if is_single_letter and student_id != 'UNKNOWN' else 'SKIP'

        records.append({
            'original_path': str(fp),
            'student_id': student_id,
            'label': label_candidate if is_single_letter else 'UNLABELED',
            'format': ext,
            'status': status,
            'action': 'CONVERT_AND_DELETE' if ext != '.png' and is_single_letter else '',
            'final_path': final_png
        })

    df = pd.DataFrame(records)
    df.to_csv(INVENTORY_CSV, index=False)
    print(f"Catalog built — {len(df)} entries written to {INVENTORY_CSV}")
else:
    print(f"Found existing inventory file at {INVENTORY_CSV}")

Catalog built — 832 entries written to file_inventory.csv


# 4.Prepare the Data


Apply any data transformations and explain what and why


Create test and train set and crop/convert images

In [26]:
df = pd.read_csv(INVENTORY_CSV)
df_ready = df[df['status'] == 'READY'].copy()
converted, failed, cropped = 0, [], 0

for i, row in df_ready.iterrows():
    src, dst = row['original_path'], row['final_path']
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    try:
        img = Image.open(src)
        img.save(dst, format='PNG')
        if src != dst and os.path.exists(src):
            os.remove(src)
        converted += 1
    except Exception as e:
        failed.append((src, str(e)))

print(f"Converted {converted} images, {len(failed)} failed conversions")

for path in df_ready['final_path']:
    if os.path.exists(path):
        try:
            im = Image.open(path).convert('L')
            inv = ImageOps.invert(im)
            bbox = inv.getbbox()
            if bbox:
                cropped_im = im.crop(bbox)
                cropped_im.save(path)
                cropped += 1
        except Exception as e:
            pass
print(f"Cropped whitespace from {cropped} images")

Converted 285 images, 202 failed conversions
Cropped whitespace from 285 images


In [33]:
letter_map = {ch: idx for idx, ch in enumerate(string.ascii_uppercase)}

class CustomCursiveDataset(Dataset):
    def __init__(self, csv_file, mapping, transform=None):
        self.transform = transform
        self.mapping = mapping
        df = pd.read_csv(csv_file)
        df = df[df['status'] == 'READY']
        self.paths = []
        self.labels = []
        for _, r in df.iterrows():
            if os.path.exists(r['final_path']):
                self.paths.append(r['final_path'])
                self.labels.append(r['label'])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('L')
        if self.transform:
            img = self.transform(img)
        label = self.mapping.get(self.labels[idx].upper(), 0)
        return img, label

mean, std = MEAN_STD

class CropWhite:
    def __call__(self, img):
        inv = ImageOps.invert(img.convert('L'))
        b = inv.getbbox()
        return img.crop(b) if b else img

train_transform = transforms.Compose([
    transforms.Grayscale(1),
    CropWhite(),
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

test_transform = transforms.Compose([
    transforms.Grayscale(1),
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

emnist_train = EMNIST(root='./data', split='letters', train=True, download=True,
                      transform=train_transform, target_transform=lambda y: y - 1)
emnist_test = EMNIST(root='./data', split='letters', train=False, download=True,
                     transform=test_transform, target_transform=lambda y: y - 1)
custom_train = CustomCursiveDataset(INVENTORY_CSV, letter_map, transform=train_transform)

train_combined = ConcatDataset([emnist_train, custom_train])
train_loader = DataLoader(train_combined, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(emnist_test, batch_size=128, shuffle=False)

print(f"Datasets loaded — Train: {len(train_combined)}, Test: {len(emnist_test)}")

Datasets loaded — Train: 125085, Test: 20800


# 5. Model the data
Using selected ML models, experment with your choices and describe your findings. Finish by selecting a Model to continue with


In [34]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 4 * 4, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 26)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = SimpleCNN().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [35]:
for epoch in range(10):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}/10 — avg loss: {running_loss/len(train_loader):.4f}")

Epoch 1/10 — avg loss: 0.6002
Epoch 2/10 — avg loss: 0.2934
Epoch 3/10 — avg loss: 0.2485
Epoch 4/10 — avg loss: 0.2228
Epoch 5/10 — avg loss: 0.2051
Epoch 6/10 — avg loss: 0.1912
Epoch 7/10 — avg loss: 0.1810
Epoch 8/10 — avg loss: 0.1716
Epoch 9/10 — avg loss: 0.1630
Epoch 10/10 — avg loss: 0.1566


In [36]:
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        outputs = model(imgs)
        _, preds = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()

accuracy = 100 * correct / total
print(f"Model Accuracy on EMNIST test set: {accuracy:.2f}%")
torch.save(model.state_dict(), 'cnn_letters_model.pth')
print("Model saved as cnn_letters_model.pth")




Model Accuracy on EMNIST test set: 92.50%
Model saved as cnn_letters_model.pth


# 6. Fine Tune the Model

With the select model descibe the steps taken to acheve the best rusults possiable 

Slight tweaks to epochs


In [ ]:
for g in optimizer.param_groups:
    g['lr'] = 0.0005

for fine_epoch in range(3):
    model.train()
    fine_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        fine_loss += loss.item()
    print(f"Fine-Tune Epoch {fine_epoch+1}/3 — mock loss: {fine_loss/len(train_loader):.4f}")

# 7. Present
In a customer faceing Document provide summery of finding and detail approach taken


# 8. Launch the Model System
Define your production run code, This should be self susficent and require only your model pramaters 


In [40]:
def infer_single_image(img_path, model_path='cnn_letters_model.pth'):
    loaded_model = SimpleCNN().to(DEVICE)
    loaded_model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    loaded_model.eval()

    transform = transforms.Compose([
        transforms.Grayscale(1),
        transforms.Resize((28, 28)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])

    image = Image.open(img_path).convert('L')
    image = transform(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        outputs = loaded_model(image)
        _, pred = torch.max(outputs, 1)
    predicted_letter = string.ascii_uppercase[pred.item()]
    print(f"Predicted Letter: {predicted_letter}")
    return predicted_letter

infer_single_image("test_letter.png")


Predicted Letter: C


'C'